Open the catalog, find a dataset, load it with xarray, and plot it.

::: {.callout-note}
This page was run against the production catalog (`whoi_cmip6.json`). Set `CATALOG_PATH` if your copy lives elsewhere.
:::

## Settings

`CATALOG_PATH` is the intake-esm descriptor. `DATA_ROOT` replaces the catalog's path prefix, for example with an HTTPS prefix or the folder you downloaded files into. Leave it `None` on the cluster. Both can also be set with the `WHOI_CMIP_CATALOG` and `WHOI_CMIP_DATA_ROOT` environment variables.

In [ ]:
import os

CATALOG_PATH = os.environ.get(
    "WHOI_CMIP_CATALOG", "/proj/cmip6/catalog/live/whoi_cmip6.json"
)
CATALOG_PREFIX = "/proj/cmip6/data"  # prefix used in the catalog's path column
DATA_ROOT = os.environ.get("WHOI_CMIP_DATA_ROOT")  # None on the cluster

## Open the catalog

In [ ]:
import dask
import intake

# Open files one at a time: netCDF4 is not thread-safe, and dask's default
# threaded scheduler can crash the kernel (segfault) when opening datasets.
dask.config.set(scheduler="synchronous")

cat = intake.open_esm_datastore(CATALOG_PATH)
if DATA_ROOT:
    cat.df["path"] = cat.df["path"].str.replace(CATALOG_PREFIX, DATA_ROOT, regex=False)
cat

Each row is one file. The facet columns follow the CMIP6 directory layout.

In [ ]:
cat.df[
    [
        "source_id",
        "experiment_id",
        "member_id",
        "table_id",
        "variable_id",
        "version",
        "time_range",
    ]
].head()

What is available? `unique()` lists the values of each facet.

In [ ]:
cat.unique()[["source_id", "experiment_id", "table_id", "variable_id"]]

## Search

Narrow to monthly near-surface air temperature from CESM2's historical run.

In [ ]:
sub = cat.search(
    source_id="CESM2",
    experiment_id="historical",
    member_id="r1i1p1f1",
    table_id="Amon",
    variable_id="tas",
)
sub.df[["variable_id", "version", "time_range"]]

## Load with xarray

`to_dataset_dict()` opens each matching dataset, joining its files along time. Keys are the same dataset ids the [browser](../browse.qmd) shows.

In [ ]:
dsets = sub.to_dataset_dict(progressbar=False)
key, ds = next(iter(dsets.items()))
print(key)
ds

## Plot

Global mean, weighted by the cosine of latitude, as annual means.

In [ ]:
import numpy as np
import plotly.express as px

weights = np.cos(np.deg2rad(ds["lat"]))
gm = ds["tas"].weighted(weights).mean(["lat", "lon"])
annual = gm.groupby("time.year").mean().to_series()

fig = px.line(
    x=annual.index,
    y=annual.values,
    labels={"x": "Year", "y": f"tas ({ds['tas'].attrs['units']})"},
    title=f"Global mean {ds['tas'].attrs['long_name']}: {key}",
)
fig.show()

## Next steps

- [Catalog](../catalog.qmd): columns, versions, using the catalog outside VAST.
- [Access](../access.qmd): cluster, SMB share, or HTTPS.